# 05 API FastAPI et Dashboard Streamlit

## Objectif
Exposer le pipeline RAG via une API REST et un dashboard interactif.

## Architecture
- **FastAPI** : backend API REST
- **Streamlit** : interface utilisateur interactive
- **Déploiement** : Hugging Face Spaces (gratuit)

## Endpoints API
| Endpoint | Description |
|----------|-------------|
| GET `/` | Statut |
| GET `/health` | Santé du système |
| POST `/query` | Question → Réponse RAG |
| GET `/metrics` | Métriques RAGAS |

In [1]:
# ============================================================
# Création du fichier api/main.py
# ============================================================

api_code = '''
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional, List
from pathlib import Path
import os
import time
import json

# LangChain
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ============================================================
# Initialisation
# ============================================================

app = FastAPI(
    title       = "Biomedical RAG API",
    description = "Question Answering system on mental health of young graduates using RAG and LLMs",
    version     = "1.0.0"
)

# Chemins
BASE_DIR   = Path(__file__).resolve().parent.parent
DATA_DIR   = BASE_DIR / "data" / "processed"
CHROMA_DIR = str(DATA_DIR / "chroma_db")

# Chargement clé API
env_path = BASE_DIR / ".env"
if env_path.exists():
    with open(env_path, "r", encoding="ascii") as f:
        for line in f:
            line = line.strip()
            if "=" in line and not line.startswith("#"):
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

# Chargement des composants
print("Chargement des composants RAG...")

embedding_model = HuggingFaceEmbeddings(
    model_name    = "all-MiniLM-L6-v2",
    model_kwargs  = {"device": "cpu"},
    encode_kwargs = {"normalize_embeddings": True}
)

vectorstore = Chroma(
    persist_directory  = CHROMA_DIR,
    embedding_function = embedding_model,
    collection_name    = "pubmed_biomedical"
)

llm = ChatGroq(
    model       = "llama-3.3-70b-versatile",
    temperature = 0.1,
    api_key     = GROQ_API_KEY
)

# Paramètres RAG
SEUIL_PERTINENCE = 0.80
K_CHUNKS         = 5
MARGE_IC         = 0.05

# Prompts
PROMPT_STRICT = ChatPromptTemplate.from_messages([
    ("system", """You are a specialized biomedical research assistant 
focused on mental health of young graduates and unemployment.

STRICT RULES:
1. Answer ONLY based on the provided PubMed context below
2. NEVER invent information not present in the context
3. Always cite the PMID and year of the sources you use
4. Structure your answer with: Main Finding, Evidence, and Limitations

CONTEXT FROM PUBMED:
{context}
"""),
    ("human", "Question: {question}\\n\\nProvide an evidence-based answer.")
])

PROMPT_FALLBACK = ChatPromptTemplate.from_messages([
    ("system", """You are a specialized biomedical research assistant.

IMPORTANT: The available PubMed context has limited relevance.
Use chain-of-thought reasoning but MUST clearly warn the user.

RULES:
1. Start with: " LIMITED CONTEXT WARNING: ..."
2. Use step-by-step reasoning
3. Recommend consulting additional sources

AVAILABLE CONTEXT:
{context}
"""),
    ("human", "Question: {question}\\n\\nReason step-by-step with appropriate caveats.")
])

# ============================================================
# Schémas
# ============================================================

class QueryInput(BaseModel):
    question : str
    k        : Optional[int] = 5

class Source(BaseModel):
    pmid      : str
    titre     : str
    journal   : str
    annee     : str
    score     : float
    url_pubmed: str

class QueryOutput(BaseModel):
    question    : str
    answer      : str
    mode        : str
    score_moyen : float
    sources     : List[Source]
    n_chunks    : int

# ============================================================
# Fonctions RAG
# ============================================================

def formater_contexte(docs_scores):
    parts = []
    for i, (doc, score) in enumerate(docs_scores):
        parts.append(f"""[Source {i+1}]
PMID    : {doc.metadata["pmid"]}
Titre   : {doc.metadata["titre"]}
Année   : {doc.metadata["annee"]}
Score   : {score:.4f}
Contenu : {doc.page_content}""")
    return "\\n---\\n".join(parts)

def extraire_sources(docs_scores):
    sources = []
    for doc, score in docs_scores:
        sources.append(Source(
            pmid       = doc.metadata["pmid"],
            titre      = doc.metadata["titre"],
            journal    = doc.metadata["journal"],
            annee      = doc.metadata["annee"],
            score      = round(score, 4),
            url_pubmed = f"https://pubmed.ncbi.nlm.nih.gov/{doc.metadata['pmid']}/"
        ))
    return sources

# ============================================================
# Endpoints
# ============================================================

@app.get("/")
def root():
    return {
        "status"     : "online",
        "model"      : "Llama 3.3 70B via Groq",
        "embedding"  : "all-MiniLM-L6-v2",
        "chunks"     : vectorstore._collection.count(),
        "description": "Biomedical RAG : Mental health of young graduates"
    }

@app.get("/health")
def health():
    try:
        test = vectorstore.similarity_search("test", k=1)
        return {
            "status"        : "healthy",
            "vectorstore"   : "connected",
            "chunks"        : vectorstore._collection.count(),
            "llm"           : "Llama 3.3 70B via Groq"
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/query", response_model=QueryOutput)
def query(data: QueryInput):
    try:
        # Recherche
        docs_scores = vectorstore.similarity_search_with_score(
            query=data.question, k=data.k
        )
        scores      = [score for _, score in docs_scores]
        score_moyen = sum(scores) / len(scores)

        # Choix du mode
        if score_moyen < SEUIL_PERTINENCE:
            mode   = "strict"
            prompt = PROMPT_STRICT
        else:
            mode   = "fallback"
            prompt = PROMPT_FALLBACK

        # Génération
        contexte = formater_contexte(docs_scores)
        chain    = prompt | llm | StrOutputParser()
        answer   = chain.invoke({
            "question": data.question,
            "context" : contexte
        })

        return QueryOutput(
            question    = data.question,
            answer      = answer,
            mode        = mode,
            score_moyen = round(score_moyen, 4),
            sources     = extraire_sources(docs_scores),
            n_chunks    = len(docs_scores)
        )

    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

@app.get("/metrics")
def metrics():
    metrics_path = DATA_DIR / "metriques_ragas.csv"
    if not metrics_path.exists():
        raise HTTPException(status_code=404, detail="Metrics not found")
    
    import pandas as pd
    df = pd.read_csv(metrics_path)
    
    return {
        "n_questions"       : len(df),
        "context_relevance" : round(df["context_relevance"].mean(), 3),
        "faithfulness"      : round(df["faithfulness"].mean(), 3),
        "answer_relevancy"  : round(df["answer_relevancy"].mean(), 3),
        "global_score"      : round(df[["context_relevance",
                                        "faithfulness",
                                        "answer_relevancy"]].mean().mean(), 3)
    }
'''

with open('../api/main.py', 'w', encoding='utf-8') as f:
    f.write(api_code)

print(" api/main.py créé")

 api/main.py créé


In [2]:
dashboard_code = '''
import streamlit as st
import requests
import json
from pathlib import Path

# ============================================================
# Configuration
# ============================================================

st.set_page_config(
    page_title = "Biomedical RAG : Mental Health",
    page_icon  = "🧠",
    layout     = "wide"
)

API_URL = "http://127.0.0.1:8000"

# ============================================================
# Fonctions utilitaires
# ============================================================

def query_api(question, k=5):
    """Envoie une question à l\'API RAG."""
    try:
        response = requests.post(
            f"{API_URL}/query",
            json    = {"question": question, "k": k},
            timeout = 60
        )
        if response.status_code == 200:
            return response.json()
        else:
            return {"error": response.text}
    except Exception as e:
        return {"error": str(e)}

def get_metrics():
    """Récupère les métriques RAGAS."""
    try:
        response = requests.get(f"{API_URL}/metrics", timeout=10)
        if response.status_code == 200:
            return response.json()
        return None
    except:
        return None

def get_health():
    """Vérifie la santé de l\'API."""
    try:
        response = requests.get(f"{API_URL}/health", timeout=5)
        return response.status_code == 200
    except:
        return False

# ============================================================
# Sidebar
# ============================================================

st.sidebar.title("🧠 Biomedical RAG")
st.sidebar.markdown("**Domaine** : Santé mentale des jeunes diplômés")
st.sidebar.markdown("**Modèle** : Llama 3.3 70B via Groq")
st.sidebar.markdown("**Embedding** : all-MiniLM-L6-v2")
st.sidebar.markdown("**Base** : 836 chunks PubMed")

st.sidebar.divider()

# Statut API
api_ok = get_health()
if api_ok:
    st.sidebar.success("✅ API connectée")
else:
    st.sidebar.error("❌ API non connectée")

st.sidebar.divider()

# Métriques RAGAS
metrics = get_metrics()
if metrics:
    st.sidebar.markdown("### 📊 Métriques RAGAS")
    st.sidebar.metric("Context Relevance",  f"{metrics[\'context_relevance\']:.3f}")
    st.sidebar.metric("Faithfulness",       f"{metrics[\'faithfulness\']:.3f}")
    st.sidebar.metric("Answer Relevancy",   f"{metrics[\'answer_relevancy\']:.3f}")
    st.sidebar.metric("Score Global",       f"{metrics[\'global_score\']:.3f}")

st.sidebar.divider()

# Paramètres
st.sidebar.markdown("### ⚙️ Paramètres")
k_chunks = st.sidebar.slider(
    "Nombre de chunks récupérés",
    min_value=3, max_value=10, value=5
)

# Navigation
page = st.sidebar.radio(
    "Navigation",
    ["💬 Question Answering", "📊 Évaluation du système", "📚 À propos"]
)

# ============================================================
# PAGE 1 : Question Answering
# ============================================================

if page == "💬 Question Answering":
    st.title("💬 Biomedical Question Answering")
    st.markdown("""
    Posez une question sur la **santé mentale des jeunes diplômés** 
    en recherche d\'emploi. Le système récupère les articles PubMed 
    les plus pertinents et génère une réponse sourcée.
    """)

    # Exemples de questions
    st.markdown("### 💡 Exemples de questions")
    exemples = [
        "What are the main mental health consequences of unemployment in young graduates?",
        "Does job insecurity cause depression in recent graduates?",
        "What is the scarring effect of unemployment on mental health?",
        "How does precarious employment affect wellbeing of educated workers?",
        "What interventions help improve mental health of unemployed graduates?"
    ]

    col1, col2 = st.columns(2)
    for i, exemple in enumerate(exemples):
        if i % 2 == 0:
            if col1.button(f"📌 {exemple[:60]}...", key=f"ex_{i}"):
                st.session_state['question'] = exemple
        else:
            if col2.button(f"📌 {exemple[:60]}...", key=f"ex_{i}"):
                st.session_state['question'] = exemple

    st.divider()

    # Zone de question
    question = st.text_area(
        "Votre question :",
        value=st.session_state.get(\'question\', \'\'),
        height=100,
        placeholder="Ex: What are the psychological effects of post-graduation unemployment?"
    )

    if st.button("🔍 Rechercher", type="primary", disabled=not api_ok):
        if question.strip():
            with st.spinner("Recherche en cours..."):
                resultat = query_api(question, k=k_chunks)

            if "error" in resultat:
                st.error(f"Erreur : {resultat[\'error\']}")
            else:
                # Mode badge
                if resultat[\'mode\'] == \'strict\':
                    st.success(f"✅ Mode Strict — Score similarité : {resultat[\'score_moyen\']:.4f}")
                else:
                    st.warning(f"⚠️ Mode Fallback — Score similarité : {resultat[\'score_moyen\']:.4f}")

                # Réponse
                st.markdown("### 📝 Réponse")
                st.markdown(resultat[\'answer\'])

                st.divider()

                # Sources
                st.markdown("### 📚 Sources PubMed utilisées")
                for i, source in enumerate(resultat[\'sources\']):
                    with st.expander(f"[{i+1}] PMID {source[\'pmid\']} ({source[\'annee\']}) — Score : {source[\'score\']:.4f}"):
                        st.markdown(f"**Titre** : {source[\'titre\']}")
                        st.markdown(f"**Journal** : {source[\'journal\']}")
                        st.markdown(f"**Lien** : [{source[\'url_pubmed\']}]({source[\'url_pubmed\']})")
        else:
            st.warning("Veuillez entrer une question.")

# ============================================================
# PAGE 2 : Évaluation
# ============================================================

elif page == " Évaluation du système":
    st.title(" Évaluation du Système RAG")
    st.markdown("""
    Résultats de l\'évaluation sur **16 questions** en mode Strict
    avec des métriques RAGAS-style.
    """)

    if metrics:
        col1, col2, col3, col4 = st.columns(4)
        col1.metric("Context Relevance",  f"{metrics[\'context_relevance\']:.3f}", 
                    delta="✅ > 0.7")
        col2.metric("Faithfulness",       f"{metrics[\'faithfulness\']:.3f}",
                    delta="⚠️ Extrapolations LLM")
        col3.metric("Answer Relevancy",   f"{metrics[\'answer_relevancy\']:.3f}",
                    delta="✅ > 0.7")
        col4.metric("Score Global",       f"{metrics[\'global_score\']:.3f}",
                    delta="✅ Bon système RAG")

        st.divider()
        st.markdown("""
        ### 🔍 Interprétation

        | Métrique | Score | Interprétation |
        |----------|-------|----------------|
        | Context Relevance | 0.837 |  Les chunks récupérés sont très pertinents |
        | Faithfulness | 0.675 |  Le LLM enrichit avec ses connaissances générales |
        | Answer Relevancy | 0.863 |  Les réponses répondent précisément aux questions |
        | **Score Global** | **0.792** |  **Bon système RAG** |

        ### 📋 Détail par mode
        - **Mode Strict** : 16/23 questions → contexte suffisant
        - **Mode Fallback** : 7/23 questions → questions hors domaine
        """)

# ============================================================
# PAGE 3 : À propos
# ============================================================

elif page == "📚 À propos":
    st.title("📚 À propos du projet")
    st.markdown("""
    ## Biomedical Question Answering with RAG and LLMs

    ### 🎯 Objectif
    Système de question-réponse biomédical spécialisé sur la 
    **santé mentale des jeunes diplômés en recherche d\'emploi**.

    ### 🏗️ Architecture
    - **652 abstracts PubMed** collectés via API (2000-2025)
    - **836 chunks** indexés dans ChromaDB
    - **Embeddings** : all-MiniLM-L6-v2 (384 dimensions)
    - **LLM** : Llama 3.3 70B via Groq API
    - **Stratégie** : RAG avec fallback (mode strict / mode chain-of-thought)

    ### 📊 Performances
    | Métrique | Score |
    |----------|-------|
    | Context Relevance | 0.837 |
    | Faithfulness | 0.675 |
    | Answer Relevancy | 0.863 |
    | **Score Global** | **0.792** |

    ### 🔗 Liens
    - **GitHub** : [Dboy003/biomedical-rag-llm](https://github.com/Dboy003/biomedical-rag-llm)
    - **API Docs** : [FastAPI Swagger](http://127.0.0.1:8000/docs)

    ### 👤 Auteur
    **Mourad DO-REGO** : Data Scientist / Biostatisticien
    """)
'''

with open('../app.py', 'w', encoding='utf-8') as f:
    f.write(dashboard_code)

print(" app.py créé")

 app.py créé


In [3]:
dockerfile = '''FROM python:3.11-slim

WORKDIR /app

COPY requirements_api.txt .
RUN pip install --no-cache-dir -r requirements_api.txt

COPY api/ ./api/
COPY data/processed/chroma_db ./data/processed/chroma_db
COPY data/processed/metriques_ragas.csv ./data/processed/metriques_ragas.csv

EXPOSE 8000

CMD ["uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000"]
'''

with open('../Dockerfile', 'w', encoding='utf-8') as f:
    f.write(dockerfile)

In [6]:
# Lire le fichier app.py
with open('../app.py', 'r', encoding='utf-8') as f:
    content = f.read()

# Remplacer l'URL locale par localhost (API et dashboard sur le même container)
content = content.replace(
    'API_URL = "http://127.0.0.1:8000"',
    'API_URL = "http://localhost:8000"'
)

with open('../app.py', 'w', encoding='utf-8') as f:
    f.write(content)

print(" app.py mis à jour")

 app.py mis à jour


## Lancement local
uvicorn api.main:app --reload --port 8000
streamlit run app.py